In [22]:
!pip install -q statsmodels scikit-learn torch plotly

In [23]:
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.preprocessing import MinMaxScaler
from warnings import simplefilter
simplefilter(action='ignore', category=FutureWarning)
simplefilter(action='ignore', category=DeprecationWarning)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode
import plotly.io as pio

init_notebook_mode(connected=True)
pio.renderers.default = 'colab'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Device: cpu


In [ ]:
STOCK_DIRECTORY = 'src/data/raw-data'  # thư mục chứa các file *.csv (thay bằng /content/data nếu chạy colab)
OUTPUT_ROOT     = 'src/data/outputs'   # thư mục lưu kết quả (thay bằng /content/outputs nếu chạy colab)
START_YEAR      = 2022                 # chỉ lấy dữ liệu từ năm này

SEQ_LEN      = 20     # số ngày nhìn lại
HIDDEN_SIZE  = 64
NUM_LAYERS   = 2
DROPOUT      = 0.2
EPOCHS       = 100
LR           = 1e-3
BATCH_SIZE   = 32
PATIENCE     = 10     # early stopping
TEST_SIZE    = 0.15
VALID_SIZE   = 0.15

os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('Cấu hình OK')
print(f'  STOCK_DIRECTORY : {STOCK_DIRECTORY}')
print(f'  OUTPUT_ROOT     : {OUTPUT_ROOT}')

Cấu hình OK
  STOCK_DIRECTORY : /content/data
  OUTPUT_ROOT     : /content/outputs


In [25]:
class StockDataset(Dataset):
    def __init__(self, X, y, seq_len):
        self.X       = torch.tensor(X, dtype=torch.float32)
        self.y       = torch.tensor(y, dtype=torch.float32)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.X) - self.seq_len

    def __getitem__(self, idx):
        return (self.X[idx : idx + self.seq_len],   # (seq_len, features)
                self.y[idx + self.seq_len])          # scalar

print('StockDataset defined')

StockDataset defined


In [26]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)            # (batch, seq_len, hidden)
        return self.fc(out[:, -1, :]).squeeze(-1)   # (batch,)

print('LSTMModel defined')

LSTMModel defined


In [27]:
def add_features(df: pd.DataFrame, shift_target: bool = True) -> pd.DataFrame:
    """
    Thêm các chỉ báo kỹ thuật.
    shift_target=True  → dùng khi train (target = giá ngày tiếp theo)
    shift_target=False → dùng khi predict tương lai (không shift close)
    """
    df = df.copy()

    # Moving averages
    df['EMA_9']  = df['close'].ewm(span=9).mean().shift()
    df['SMA_5']  = df['close'].rolling(5).mean().shift()
    df['SMA_10'] = df['close'].rolling(10).mean().shift()
    df['SMA_15'] = df['close'].rolling(15).mean().shift()
    df['SMA_30'] = df['close'].rolling(30).mean().shift()

    # RSI
    close  = df['close']
    delta  = close.diff().iloc[1:]
    up     = delta.clip(lower=0)
    down   = delta.clip(upper=0).abs()
    rs     = up.rolling(14).mean() / down.rolling(14).mean()
    rsi    = 100.0 - (100.0 / (1.0 + rs))
    df['RSI'] = rsi.reindex(df.index).fillna(0)

    # MACD
    ema12 = close.ewm(span=12, min_periods=12).mean()
    ema26 = close.ewm(span=26, min_periods=26).mean()
    df['MACD']        = ema12 - ema26
    df['MACD_signal'] = df['MACD'].ewm(span=9, min_periods=9).mean()

    if shift_target:
        df['close'] = df['close'].shift(-1)   # target = giá đóng cửa ngày sau
        df = df.iloc[33:-1]                   # bỏ NaN đầu & dòng cuối
    else:
        df = df.iloc[33:]                     # chỉ bỏ NaN đầu

    df.index = range(len(df))
    return df

print('add_features defined')

add_features defined


In [28]:
def train_stock(stock_code: str, csv_path: str, out_dir: str, verbose: bool = True):
    """Train LSTM cho một mã và lưu toàn bộ artifacts vào out_dir."""
    os.makedirs(out_dir, exist_ok=True)

    # ── 1. Load & lọc ────────────────────────────────────────
    df = pd.read_csv(csv_path, sep=',')
    df['time'] = pd.to_datetime(df['time'])
    df = df[df['time'].dt.year >= START_YEAR].copy()
    df.index = range(len(df))

    if len(df) < 100:
        print(f'Bỏ qua {stock_code}: không đủ dữ liệu ({len(df)} dòng)')
        return None

    # ── 2. Features ──────────────────────────────────────────
    df = add_features(df, shift_target=True)

    # ── 3. Split ─────────────────────────────────────────────
    n           = len(df)
    test_split  = int(n * (1 - TEST_SIZE))
    valid_split = int(n * (1 - TEST_SIZE - VALID_SIZE))

    train_df = df.loc[:valid_split].copy()
    valid_df = df.loc[valid_split+1 : test_split].copy()
    test_df  = df.loc[test_split+1:].copy()

    # ── 4. X / y ─────────────────────────────────────────────
    drop_cols    = ['time']
    feature_cols = [c for c in train_df.columns if c not in drop_cols + ['close']]

    def split_xy(d):
        return d[feature_cols].values, d['close'].values

    X_tr, y_tr = split_xy(train_df)
    X_va, y_va = split_xy(valid_df)
    X_te, y_te = split_xy(test_df)

    # ── 5. Scale ─────────────────────────────────────────────
    scaler_X = MinMaxScaler()
    X_tr = scaler_X.fit_transform(X_tr)
    X_va = scaler_X.transform(X_va)
    X_te = scaler_X.transform(X_te)

    scaler_y = MinMaxScaler()
    y_tr = scaler_y.fit_transform(y_tr.reshape(-1,1)).flatten()
    y_va = scaler_y.transform(y_va.reshape(-1,1)).flatten()
    y_te = scaler_y.transform(y_te.reshape(-1,1)).flatten()

    # ── 6. DataLoaders ───────────────────────────────────────
    def make_loader(X, y):
        return DataLoader(StockDataset(X, y, SEQ_LEN),
                          batch_size=BATCH_SIZE, shuffle=False)

    train_loader = make_loader(X_tr, y_tr)
    valid_loader = make_loader(X_va, y_va)
    test_loader  = make_loader(X_te, y_te)

    # ── 7. Model / optimizer ─────────────────────────────────
    model = LSTMModel(
        input_size=len(feature_cols),
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5)

    best_val, patience_cnt = float('inf'), 0
    history = {'train': [], 'valid': []}
    best_path = os.path.join(out_dir, 'best_lstm.pt')

    # ── 8. Training loop ─────────────────────────────────────
    for epoch in range(1, EPOCHS+1):
        model.train()
        tr_losses = []
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_losses.append(loss.item())

        model.eval()
        va_losses = []
        with torch.no_grad():
            for Xb, yb in valid_loader:
                va_losses.append(
                    criterion(model(Xb.to(DEVICE)), yb.to(DEVICE)).item())

        tr_loss = np.mean(tr_losses)
        va_loss = np.mean(va_losses)
        scheduler.step(va_loss)
        history['train'].append(tr_loss)
        history['valid'].append(va_loss)

        if verbose and epoch % 20 == 0:
            print(f'    Epoch {epoch:3d} | Train {tr_loss:.6f} | Valid {va_loss:.6f}')

        if va_loss < best_val:
            best_val = va_loss
            torch.save(model.state_dict(), best_path)
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                if verbose: print(f'    Early stopping tại epoch {epoch}')
                break

    # ── 9. Evaluate trên test set ────────────────────────────
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    model.eval()
    y_pred_sc, y_true_sc = [], []
    with torch.no_grad():
        for Xb, yb in test_loader:
            y_pred_sc.extend(model(Xb.to(DEVICE)).cpu().numpy())
            y_true_sc.extend(yb.numpy())

    y_pred = scaler_y.inverse_transform(
        np.array(y_pred_sc).reshape(-1,1)).flatten()
    y_true = scaler_y.inverse_transform(
        np.array(y_true_sc).reshape(-1,1)).flatten()

    mse  = mean_squared_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)

    # ── 10. Lưu artifacts ───────────────────────────────────
    with open(os.path.join(out_dir, 'scaler_X.pkl'), 'wb') as f:
        pickle.dump(scaler_X, f)
    with open(os.path.join(out_dir, 'scaler_y.pkl'), 'wb') as f:
        pickle.dump(scaler_y, f)

    meta = {
        'stock_code':      stock_code,
        'feature_cols':    feature_cols,
        'seq_len':         SEQ_LEN,
        'hidden_size':     HIDDEN_SIZE,
        'num_layers':      NUM_LAYERS,
        'dropout':         DROPOUT,
        'input_size':      len(feature_cols),
        'valid_split_idx': int(valid_split),
        'test_split_idx':  int(test_split),
    }
    with open(os.path.join(out_dir, 'meta.json'), 'w') as f:
        json.dump(meta, f, indent=2)
    with open(os.path.join(out_dir, 'metrics.json'), 'w') as f:
        json.dump({'mse': float(mse), 'mape': float(mape)}, f, indent=2)

    # predictions.csv
    test_time = test_df['time'].reset_index(drop=True).iloc[SEQ_LEN:].reset_index(drop=True)
    pd.DataFrame({'time': test_time, 'y_true': y_true, 'y_pred': y_pred}).to_csv(
        os.path.join(out_dir, 'predictions.csv'), index=False)

    # loss curve
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(history['train'], label='Train loss')
    ax.plot(history['valid'], label='Valid loss')
    ax.set_title(f'{stock_code} — Loss curve')
    ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
    ax.legend(); plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'loss_curve.png'), dpi=100)
    plt.close()

    # prediction plot
    all_y = scaler_y.inverse_transform(
        np.concatenate([y_tr, y_va, y_te]).reshape(-1,1)).flatten()
    fig, axes = plt.subplots(2, 1, figsize=(14, 7))
    axes[0].plot(df['time'].values[:len(all_y)], all_y,
                 color='steelblue', label='Truth', alpha=0.5)
    axes[0].plot(test_time.values, y_pred, color='purple', label='Prediction')
    axes[0].set_title(f'{stock_code} — Full series'); axes[0].legend()
    axes[1].plot(test_time.values, y_true, color='steelblue', label='Truth')
    axes[1].plot(test_time.values, y_pred, color='purple',    label='Prediction')
    axes[1].set_title(f'{stock_code} — Test zoom | MAPE={mape:.4f}'); axes[1].legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, 'prediction_plot.png'), dpi=100)
    plt.close()

    return {'stock': stock_code, 'mse': mse, 'mape': mape}

print('train_stock defined')

train_stock defined


In [29]:
def load_artifacts(stock_code: str):
    """Load model + scalers + meta đã train."""
    out_dir = os.path.join(OUTPUT_ROOT, stock_code)
    if not os.path.isdir(out_dir):
        raise FileNotFoundError(
            f'Không tìm thấy output cho {stock_code}. Chạy train trước.')

    with open(os.path.join(out_dir, 'meta.json'))        as f: meta     = json.load(f)
    with open(os.path.join(out_dir, 'scaler_X.pkl'),'rb') as f: scaler_X = pickle.load(f)
    with open(os.path.join(out_dir, 'scaler_y.pkl'),'rb') as f: scaler_y = pickle.load(f)

    model = LSTMModel(
        input_size=meta['input_size'],
        hidden_size=meta['hidden_size'],
        num_layers=meta['num_layers'],
        dropout=meta['dropout'],
    ).to(DEVICE)
    model.load_state_dict(
        torch.load(os.path.join(out_dir, 'best_lstm.pt'), map_location=DEVICE))
    model.eval()
    return model, scaler_X, scaler_y, meta


def predict_future(stock_code: str, n_future: int = 5, csv_path: str = None):
    """
    Dự đoán n_future ngày làm việc tiếp theo.
    csv_path: tuỳ chọn — mặc định lấy từ STOCK_DIRECTORY.
    """
    model, scaler_X, scaler_y, meta = load_artifacts(stock_code)
    seq_len      = meta['seq_len']
    feature_cols = meta['feature_cols']

    if csv_path is None:
        csv_path = os.path.join(STOCK_DIRECTORY, stock_code + '.csv')

    raw = pd.read_csv(csv_path, sep=',')
    raw['time'] = pd.to_datetime(raw['time'])
    raw = raw[raw['time'].dt.year >= START_YEAR].copy()
    raw.index = range(len(raw))

    df_feat = add_features(raw, shift_target=False)
    X_all   = scaler_X.transform(df_feat[feature_cols].values)
    window  = X_all[-seq_len:].copy()

    close_idx = (feature_cols.index('close')
                 if 'close' in feature_cols else None)

    preds_scaled = []
    with torch.no_grad():
        for _ in range(n_future):
            x_t   = torch.tensor(window[np.newaxis], dtype=torch.float32).to(DEVICE)
            p_sc  = model(x_t).cpu().item()
            preds_scaled.append(p_sc)
            new_row = window[-1].copy()
            if close_idx is not None:
                new_row[close_idx] = p_sc
            window = np.vstack([window[1:], new_row])

    preds_orig = scaler_y.inverse_transform(
        np.array(preds_scaled).reshape(-1,1)).flatten()

    last_date    = df_feat['time'].iloc[-1]
    future_dates = pd.bdate_range(
        start=last_date + pd.Timedelta(days=1), periods=n_future)

    result = pd.DataFrame({'time': future_dates,
                           'predicted_close': preds_orig})

    # Lưu
    out_dir   = os.path.join(OUTPUT_ROOT, stock_code)
    save_path = os.path.join(out_dir, f'future_{n_future}days.csv')
    result.to_csv(save_path, index=False)

    return result, raw, last_date

print('load_artifacts & predict_future defined')

load_artifacts & predict_future defined


In [30]:
csv_files = sorted([f for f in os.listdir(STOCK_DIRECTORY) if f.endswith('.csv')])
print(f'Tìm thấy {len(csv_files)} mã: {[f[:-4] for f in csv_files]}\n')

summary = []
for fname in csv_files:
    code    = fname[:-4]
    csv_path = os.path.join(STOCK_DIRECTORY, fname)
    out_dir  = os.path.join(OUTPUT_ROOT, code)
    print(f'\n{'='*55}')
    print(f'  Đang train: {code}')
    print(f'{'='*55}')
    try:
        result = train_stock(code, csv_path, out_dir, verbose=True)
        if result:
            summary.append(result)
            print(f'MSE={result["mse"]:.4f} | MAPE={result["mape"]:.4f}')
    except Exception as e:
        print(f'Lỗi: {e}')
        summary.append({'stock': code, 'mse': None, 'mape': None})

# Tổng kết
sum_df = pd.DataFrame(summary).set_index('stock')
sum_df.to_csv(os.path.join(OUTPUT_ROOT, 'summary.csv'))
print('\n' + '='*55)
print('  TỔNG KẾT')
print('='*55)
display(sum_df)

Tìm thấy 32 mã: ['ACB', 'BID', 'BVH', 'CTG', 'DGC', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG', 'KDH', 'MBB', 'MSN', 'MWG', 'NVL', 'PDR', 'PLX', 'PNJ', 'POW', 'SAB', 'SSI', 'STB', 'TCB', 'TPB', 'VCB', 'VHM', 'VIC', 'VJC', 'VNM', 'VPB', 'VPL', 'VRE']


  Đang train: ACB
    Early stopping tại epoch 15
MSE=0.4446 | MAPE=0.0213

  Đang train: BID
    Early stopping tại epoch 18
MSE=10.2566 | MAPE=0.0427

  Đang train: BVH
    Epoch  20 | Train 0.007092 | Valid 0.014246
    Epoch  40 | Train 0.004791 | Valid 0.010299
    Epoch  60 | Train 0.003481 | Valid 0.008110
    Epoch  80 | Train 0.003161 | Valid 0.007566
    Early stopping tại epoch 85
MSE=44.0431 | MAPE=0.0686

  Đang train: CTG
    Early stopping tại epoch 15
MSE=7.3754 | MAPE=0.0587

  Đang train: DGC
    Epoch  20 | Train 0.003583 | Valid 0.002807
    Epoch  40 | Train 0.003364 | Valid 0.002468
    Early stopping tại epoch 58
MSE=34.3284 | MAPE=0.0590

  Đang train: FPT
    Epoch  20 | Train 0.001846 | Valid 0.002662
    Early stopping t

,mse,mape
stock,,
ACB,0.444601,0.021278
BID,10.256615,0.042668
BVH,44.043068,0.068619
CTG,7.375390,0.058685
DGC,34.328442,0.059008
FPT,16.067770,0.032143
GAS,95.938202,0.067670
GVR,7.002549,0.057611
HDB,8.928576,0.099563


In [31]:
STOCK_CODE = 'VRE'   # ← đổi mã tại đây

out_dir   = os.path.join(OUTPUT_ROOT, STOCK_CODE)
pred_path = os.path.join(out_dir, 'predictions.csv')
metr_path = os.path.join(out_dir, 'metrics.json')

pred_df = pd.read_csv(pred_path, parse_dates=['time'])
with open(metr_path) as f:
    m = json.load(f)

print(f'\n── Kết quả test set: {STOCK_CODE} ──')
print(f'  MSE  = {m["mse"]:.4f}')
print(f'  MAPE = {m["mape"]:.4f}  ({m["mape"]*100:.2f}%)')
print(f'  Số điểm test: {len(pred_df)}')
display(pred_df.tail(10))

fig = make_subplots(rows=1, cols=1)
fig.add_trace(go.Scatter(x=pred_df['time'], y=pred_df['y_true'],
                         name='Thực tế', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=pred_df['time'], y=pred_df['y_pred'],
                         name='Dự đoán', line=dict(color='purple')))
fig.update_layout(title=f'{STOCK_CODE} — Test set | MAPE={m["mape"]:.4f}',
                  xaxis_title='Ngày', yaxis_title='Giá đóng cửa')
fig.show()


── Kết quả test set: VRE ──
  MSE  = 5.4190
  MAPE = 0.0503  (5.03%)
  Số điểm test: 135


,time,y_true,y_pred
125,2026-04-14,29.300000,27.454306
126,2026-04-15,29.600000,27.605204
127,2026-04-16,28.600000,27.913670
128,2026-04-17,29.449999,28.323563
129,2026-04-20,29.300000,28.610064
130,2026-04-21,29.950000,28.786692
131,2026-04-22,29.050000,28.934965
132,2026-04-23,28.800000,29.085917
133,2026-04-24,30.800000,29.234762
134,2026-04-28,32.300000,29.318422


In [32]:
STOCK_CODE = 'VRE'   # ← đổi mã tại đây
N_FUTURE   = 5       # ← số ngày muốn dự đoán

result, raw, last_date = predict_future(STOCK_CODE, n_future=N_FUTURE)

print(f'\n── Dự đoán {N_FUTURE} ngày tiếp theo: {STOCK_CODE} ──')
display(result)

# Biểu đồ: giá thực tế gần đây + dự đoán
last_n  = min(120, len(raw))
recent  = raw.tail(last_n)

fig = go.Figure()
fig.add_trace(go.Scatter(x=recent['time'], y=recent['close'],
                         name='Giá thực tế', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=result['time'], y=result['predicted_close'],
                         name='Dự đoán tương lai',
                         mode='lines+markers',
                         line=dict(color='tomato', dash='dash')))
fig.add_vline(x=str(last_date), line_dash='dot', line_color='gray')
fig.update_layout(
    title=f'{STOCK_CODE} — Dự đoán {N_FUTURE} ngày tiếp theo',
    xaxis_title='Ngày', yaxis_title='Giá đóng cửa')
fig.show()

save_path = os.path.join(OUTPUT_ROOT, STOCK_CODE, f'future_{N_FUTURE}days.csv')
print(f'\nĐã lưu CSV: {save_path}')


── Dự đoán 5 ngày tiếp theo: VRE ──


,time,predicted_close
0,2026-04-30,29.754692
1,2026-05-01,30.095067
2,2026-05-04,30.365517
3,2026-05-05,30.556951
4,2026-05-06,30.681983



Đã lưu CSV: /content/outputs/VRE/future_5days.csv


In [33]:
summary_path = os.path.join(OUTPUT_ROOT, 'summary.csv')
if os.path.exists(summary_path):
    sum_df = pd.read_csv(summary_path, index_col='stock')
    sum_df['mape_%'] = (sum_df['mape'] * 100).round(2)
    display(sum_df.sort_values('mape'))

    # Bar chart MAPE
    fig = go.Figure(go.Bar(
        x=sum_df.index,
        y=sum_df['mape_%'],
        marker_color='MediumPurple',
        text=sum_df['mape_%'].round(2),
        textposition='outside'
    ))
    fig.update_layout(title='MAPE (%) theo mã cổ phiếu',
                      xaxis_title='Mã', yaxis_title='MAPE (%)')
    fig.show()
else:
    print('Chưa có summary.csv — chạy Cell 10 trước.')

,mse,mape,mape_%
stock,,,
HPG,0.547321,0.021022,2.10
ACB,0.444601,0.021278,2.13
TPB,0.272893,0.023901,2.39
MSN,6.163522,0.024244,2.42
VNM,5.967037,0.026658,2.67
SAB,3.951561,0.027262,2.73
FPT,16.067770,0.032143,3.21
VCB,15.483319,0.032144,3.21
VPB,1.292357,0.032405,3.24
